# Rotax 914 ULF - Step 0: Prepare the data

**In one sentence:** Checks the simulated flight data is complete and consistent, then puts every input on the same scale so no reading dominates just because its numbers are bigger.

| Goes in | Comes out |
|---|---|
| Raw simulated flights, one row per second | A scaler every later step uses |



Run the cells top to bottom with **Shift+Enter**. 

### 0. Setup

In [ ]:
# Finds the project folder wherever Jupyter was started, then loads everything.
import os, sys
d = os.path.abspath('.')
for _ in range(6):
    if os.path.exists(os.path.join(d, 'nb_helpers.py')): break
    if os.path.exists(os.path.join(d, 'validation_v4', 'nb_helpers.py')):
        d = os.path.join(d, 'validation_v4'); break
    d = os.path.dirname(d)
sys.path.insert(0, d)
from nb_helpers import *

ENGINE, PHASE = '914', 'scalers'
setup(ENGINE, PHASE)

### 1. What is in the dataset
Each simulated flight goes entirely into one group, so the final score is
measured on flights the models never saw.

In [ ]:
desc = C.describe(ENGINE)
flights = {s: len(json.load(open(os.path.join(C.data_dir(ENGINE), f"index_{s}.json"))))
           for s in ("train", "val", "test")}
overview = pd.DataFrame({"Flights": flights, "Rows (1 per second)": desc["splits"],
                         "128-second windows": desc["windows"]})
overview.index = ["Train (learn from)", "Validation (pick the best)", "Test (final score)"]
overview

### 2. The 29 inputs every model sees
Only readings a real aircraft actually provides - nothing the simulator knows
but an airframe would not.

In [ ]:
groups = [("Flight condition", P.FLIGHT_COLS, "What the aircraft is doing, and the air it flies in"),
          ("Engine sensors", P.MEASURED_COLS, "The 12 readings a real engine monitor provides"),
          ("Derived", P.DERIVED_COLS, "Computed on board from the readings above"),
          ("Twin residuals", P.RESIDUAL_COLS, "Reading minus what a healthy engine would read right now")]
pd.DataFrame([{"Group": g, "Count": len(c), "Why it is there": w,
               "Inputs": ", ".join(pretty(x) for x in c)} for g, c, w in groups])

### 3. Data check - run this before trusting anything else
Every flight listed in the index must be found in the data files with exactly
the right number of rows. If this fails, stop: training would silently skip
part of the data.

In [ ]:
problems = 0
for split in ("train", "val", "test"):
    entries = json.load(open(os.path.join(C.data_dir(ENGINE), f"index_{split}.json")))
    sample = entries[::max(1, len(entries) // 40)][:40]
    ok = sum(pq.read_table(os.path.join(C.data_dir(ENGINE), split, e["file"]), columns=["scenario_id"],
                           filters=[("scenario_id", "=", e["scenario_id"])]).num_rows == e["n_rows"]
             for e in sample)
    problems += len(sample) - ok
    print(f"{split:5s}: {ok}/{len(sample)} sampled flights found with the right number of rows")
print("\nDATA OK" if problems == 0 else "\nPROBLEM - do not train until this is fixed")

### 4. Put every input on the same scale
Fitted on the training flights only - using the test flights here would leak
the answers into training and make every score look better than it is.

In [ ]:
R.phase_scalers(ENGINE, force=False)
print("Input scaler:      ", C.scaler_path(ENGINE))
print("Life-stage scaler: ", C.aux_scaler_path(ENGINE))

### 5. Did it work?
After scaling, every input should average about 0 with a spread of about 1.

In [ ]:
import joblib
sc = joblib.load(C.scaler_path(ENGINE))
z = sc.transform(load_rows(ENGINE, P.FEATURE_COLS, split="train").to_numpy(np.float64))
fig, ax = plt.subplots(1, 2, figsize=(10, 6), sharey=True)
yy = np.arange(len(P.FEATURE_COLS))
ax[0].barh(yy, z.mean(0), color=HEALTHY); ax[0].axvline(0, color="k", lw=0.8)
ax[0].set_title("Average after scaling (should be about 0)")
ax[1].barh(yy, z.std(0), color=HEALTHY); ax[1].axvline(1, color="k", lw=0.8)
ax[1].set_title("Spread after scaling (should be about 1)")
ax[0].set_yticks(yy, [pretty(c) for c in P.FEATURE_COLS]); ax[0].invert_yaxis()
plt.tight_layout(); plt.show()

### What this means

In [ ]:
print(f"Largest average after scaling: {np.abs(z.mean(0)).max():.2f}. Small values like this are "
      f"normal - the check uses one data file, the scaler was fitted on two million rows.")
print("The data is ready. Next: step 1, 'Is anything wrong?'")

---
### Terms used

| Term | Meaning |
|---|---|
| **Window** | 128 seconds of flight data - the unit every model looks at. |
| **Twin** | A perfectly healthy simulated copy of the engine, flown through the exact same flight. |
| **Residual** | Real reading minus the healthy twin's reading. Near zero means the engine behaves as it should. |
| **Train / validation / test** | Flights used to learn / to pick the best version / to score it. A flight is only ever in one of them, so the score is on flights the model never saw. |
| **TBO** | Time between overhauls - the engine's scheduled life: 2000 h, or 1200 h for the 915. |
| **Warm start** | A step begins from what an earlier step already learned instead of from nothing. |